# 03 — Hypothesis Testing
Test three business hypotheses: (H1) premium hours run more steadily, (H2) quality tier relates to time of day, (H3) premium output persists in streaks.

In [1]:
#libraries

import pandas as pd
import seaborn as sns
import scipy.stats as st
from scipy.stats import ttest_ind
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests
from scipy.stats import chi2_contingency
from scipy.stats.contingency import association
import numpy as np
from sqlalchemy import create_engine
import os
from dotenv import load_dotenv

In [2]:
load_dotenv()

False

In [ ]:
engine = create_engine(
    f"mysql+pymysql://{os.getenv('DB_USER')}:{os.getenv('DB_PASS')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)

## H1 — When the output is better/premium,  the plant runs more steadily/more stability
During periods of premium quality output, the plant's operating conditions (chemical reagent flows, water levels, airflow, pH, etc.) don't fluctuate as much from minute to minute as they do during lower-grade periods.

**H₀: Stability of operating conditions when output is premium = stability with non-premium output**

H₁: Stability of operating conditions when output is premium ≠ stability with non-premium output

In [4]:
df = pd.read_sql("""
SELECT
       quality_tier,
       std_ph,
       std_starch_flow,
       std_amina_flow,
       (std_airflow_col01 + std_airflow_col02 + std_airflow_col03 + std_airflow_col04 + std_airflow_col05 + std_airflow_col06 + std_airflow_col07) / 7.0 AS avg_std_airflow,
       (std_level_col01 + std_level_col02 + std_level_col03 + std_level_col04 + std_level_col05 + std_level_col06 + std_level_col07) / 7.0 AS avg_std_level
FROM hourly_metrics
WHERE quality_tier IS NOT NULL;
""", engine)

In [5]:
df[df.isnull().any(axis=1)]

,quality_tier,std_ph,std_starch_flow,std_amina_flow,avg_std_airflow,avg_std_level
2640,standard,NaN,NaN,NaN,NaN,NaN
2641,premium,NaN,NaN,NaN,NaN,NaN
2642,premium,NaN,NaN,NaN,NaN,NaN
2696,premium,NaN,NaN,NaN,NaN,NaN


In [6]:
df = df.dropna()

In [6]:
df.describe()

In [7]:
df['quality_tier'].value_counts()

quality_tier
premium     2041
standard    1328
low          724
Name: count, dtype: int64

In [8]:
df.select_dtypes(include='number').skew()

std_ph             1.978143
std_starch_flow    0.498529
std_amina_flow     1.464409
avg_std_airflow    5.651022
avg_std_level      1.515313
dtype: float64

In [12]:
variables = ['std_ph', 'std_starch_flow', 'std_amina_flow','avg_std_airflow', 'avg_std_level']

for a in variables:
    premium_hp = df[df["quality_tier"] == "premium"][a]
    non_premiumhp = df[df["quality_tier"] != "premium"][a]
    stat, p = mannwhitneyu(premium_hp, non_premiumhp, alternative='two-sided')
    print(f"{a}: p = {p}")


std_ph: p = 0.021589259506100467
std_starch_flow: p = 0.5929348642889238
std_amina_flow: p = 0.5441000430795939
avg_std_airflow: p = 0.04217614859445124
avg_std_level: p = 0.07706602541793647


**Result** — **Fail to reject H₀** — operating stability is not a reliable differentiator of premium hours.

## H2 — The time-of-day bucket and the quality tier are related.

Does the plant tends to produce more top-grade output during evening and night hours than during morning and afternoon?

H₀: Tier distribution is the same across all time-of-day buckets (no association).

**H₁: Tier distribution differs across time-of-day buckets (association exists).**

In [25]:
df1 = pd.read_sql("""
SELECT
	t.time_of_day_bucket,
	h.quality_tier,
    COUNT(*) AS hours_count,
    COUNT(*) * 100.0 / SUM(COUNT(*)) OVER () AS pct_total,
    COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (PARTITION BY t.time_of_day_bucket) AS pct_bucket
FROM hourly_metrics h
JOIN time_dim t ON h.hour_id = t.hour_id
GROUP BY t.time_of_day_bucket, h.quality_tier;""", engine)

In [26]:
df1

,time_of_day_bucket,quality_tier,hours_count,pct_total,pct_bucket
0,afternoon,standard,363,8.86014,35.38012
1,afternoon,premium,482,11.76471,46.97856
2,afternoon,low,181,4.41787,17.64133
3,evening,low,175,4.27142,17.05653
4,evening,premium,551,13.44887,53.70370
5,evening,standard,300,7.32243,29.23977
6,morning,premium,486,11.86234,47.64706
7,morning,standard,341,8.32316,33.43137
8,morning,low,193,4.71076,18.92157
9,night,premium,525,12.81425,51.21951


In [ ]:
df1.describe()

In [32]:
contingency = df1.pivot(index='time_of_day_bucket', columns='quality_tier', values='hours_count')

quality_tier,low,premium,standard
time_of_day_bucket,,,
afternoon,181,482,363
evening,175,551,300
morning,193,486,341
night,175,525,325


In [34]:
chi2, p_value, dof, expected = chi2_contingency(contingency)

print("Chi-square statistic:", chi2)
print("p-value:", p_value)

Chi-square statistic: 13.90778980844903
p-value: 0.030683335951515383


In [36]:
cramers_v = association(contingency, method='cramer')
print("Cramér's V:", cramers_v)

Cramér's V: 0.041198469204982685


**Result — H2:** Chi-squared is significant (p = 0.030), so tier and time-of-day are associatedbut Cramér's V = 0.041 means the link is practically negligible. **Reject H₀**: Statistically real, practically weak.

## H3 — Good performance tends to come in stretches.

When the plant is producing top-grade output, it tends to keep doing so for several hours in a row, rather than flipping between quality levels each hour.

H₀: Premium output does NOT persist longer than other quality tiers. When premium occurs, it lasts about the same amount of time as standard or low quality periods.

**H₁: Premium output persists LONGER than other quality tiers. When premium occurs, it tends to last for several hours — longer than standard or low quality periods.**

In [4]:
df2 = pd.read_sql("""
SELECT
    streak_id,
    quality_tier,
    streak_length
FROM streak_summary;""", engine)

In [8]:
df2.isna().sum()

streak_id        0
quality_tier     0
streak_length    0
dtype: int64

In [9]:
df2

,streak_id,quality_tier,streak_length
0,0.0,premium,6
1,1.0,standard,2
2,2.0,premium,2
3,3.0,standard,2
4,4.0,premium,5
...,...,...,...
1071,1071.0,premium,1
1072,1072.0,standard,4
1073,1073.0,premium,1
1074,1074.0,standard,6


In [ ]:
df2.describe()

In [11]:
premium_streak = df2[df2["quality_tier"] == "premium"]['streak_length']
non_premium_streak = df2[df2["quality_tier"] != "premium"]['streak_length']

stat, p = mannwhitneyu(premium_streak, non_premium_streak, alternative='two-sided')

print(f"p = {p}")

p = 2.2870970611247486e-12


In [13]:
n1, n2 = len(premium_streak), len(non_premium_streak)
r = 1 - (2 * stat) / (n1 * n2)

r

np.float64(-0.24602538360105486)

**Result — H3:** Mann-Whitney p < 0.001 with a moderate effect: premium streaks (median 3 h, mean 5.27 h) are genuinely longer than non-premium runs. **Reject H₀**